# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
import hashlib
import inspect
import re
import shutil
import tempfile
from pathlib import Path

import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42


## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [ ]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_your_username"
MLFLOW_PASSWORD = "your_mlflow_password"
EXPERIMENT_NAME = "your_experiment_name"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [3]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"


def file_sha256(path: Path) -> str:
    """Calculate a stable file hash so the exact dataset version can be traced."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def find_feature_file(feature_dir: Path, dataset_version: str) -> tuple[Path, str]:
    """Find the HW01 feature dataset, preferring Parquet over CSV."""
    expected_parquet = feature_dir / f"listing_availability_features_{dataset_version}.parquet"
    expected_csv = feature_dir / f"listing_availability_features_{dataset_version}.csv"

    if expected_parquet.exists():
        return expected_parquet, "parquet"
    if expected_csv.exists():
        return expected_csv, "csv"

    parquet_candidates = sorted(feature_dir.glob("listing_availability_features_*.parquet"))
    csv_candidates = sorted(feature_dir.glob("listing_availability_features_*.csv"))

    if parquet_candidates:
        fallback = parquet_candidates[0]
        print(f"Expected dataset not found. Using available Parquet file: {fallback}")
        return fallback, "parquet"

    if csv_candidates:
        fallback = csv_candidates[0]
        print(f"Expected dataset not found. Using available CSV file: {fallback}")
        return fallback, "csv"

    raise FileNotFoundError(
        f"No HW01 feature dataset found in {feature_dir.resolve()}. "
        "Copy the CSV or Parquet output from HW01 into data/features first."
    )


dataset_path, dataset_format = find_feature_file(FEATURE_DIR, DATASET_VERSION)

if dataset_format == "parquet":
    feature_df = pd.read_parquet(dataset_path)
else:
    feature_df = pd.read_csv(dataset_path)

if metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    # If the exact metadata filename is not present, try to find any matching metadata file.
    metadata_candidates = sorted(FEATURE_DIR.glob("listing_availability_features_*_metadata.json"))
    if metadata_candidates:
        with metadata_candidates[0].open("r", encoding="utf-8") as f:
            metadata = json.load(f)
        print(f"Using metadata file: {metadata_candidates[0]}")
    else:
        metadata = {}

metadata = dict(metadata)
metadata.update(
    {
        "dataset_path": str(dataset_path),
        "dataset_format": dataset_format,
        "dataset_sha256": file_sha256(dataset_path),
        "dataset_rows": int(feature_df.shape[0]),
        "dataset_columns": int(feature_df.shape[1]),
        "notebook_dataset_version": DATASET_VERSION,
    }
)

print("Loaded dataset:", dataset_path)
print("Format:", dataset_format)
print("Shape:", feature_df.shape)
feature_df.head()


Expected dataset not found. Using available Parquet file: data/features/listing_availability_features_v1_student.parquet
Using metadata file: data/features/listing_availability_features_v1_student_metadata.json
Loaded dataset: data/features/listing_availability_features_v1_student.parquet
Format: parquet
Shape: (10480, 34)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews_before_cutoff
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student,1


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

if TARGET_COL not in feature_df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' was not found in the dataset.")

y = feature_df[TARGET_COL].astype(int)

# Clean models must not use labels, future-window fields, or audit/entity identifiers.
clean_feature_cols = [
    col
    for col in feature_df.columns
    if col not in FORBIDDEN_MODEL_COLUMNS
    and not col.startswith("future_")
]

X_clean = feature_df[clean_feature_cols].copy()

if X_clean.empty:
    raise ValueError("X_clean has no columns. Check the forbidden-column list.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)


Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 27
['room_type', 'property_type', 'neighbourhood_name', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d', 'has_reviews_before_cutoff']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

if LEAKAGE_COLUMN not in feature_df.columns:
    raise KeyError(
        f"Leakage column '{LEAKAGE_COLUMN}' was not found. "
        "HW01 should have created this future-window label helper column."
    )

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]
leaky_feature_cols = [col for col in leaky_feature_cols if col != TARGET_COL]

X_leaky = feature_df[leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)


Leaky feature count: 28
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())


Train shape: (8384, 27)
Test shape: (2096, 27)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [7]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def infer_feature_types(X: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Split dataframe columns into numeric and categorical feature lists."""
    numeric = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical = [col for col in X.columns if col not in numeric]
    return numeric, categorical


def make_preprocessor(X: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    """Build preprocessing for the exact feature set used by a run."""
    numeric, categorical = infer_feature_types(X)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric),
            ("categorical", categorical_transformer, categorical),
        ],
        remainder="drop",
    )

    return preprocessor, numeric, categorical


preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_clean)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


Numeric columns: 24
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [8]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    y_score = get_positive_scores(model, X_test)
    y_pred = (y_score >= threshold).astype(int)

    try:
        roc_auc = roc_auc_score(y_test, y_score)
    except ValueError:
        roc_auc = np.nan

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc,
    }

    return metrics, y_pred, y_score


## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [9]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def safe_name(name: str) -> str:
    """Make a safe folder name from a run name."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def save_json(path: Path, payload: dict | list) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    run_artifact_dir = ARTIFACT_DIR / safe_name(run_name)
    if run_artifact_dir.exists():
        shutil.rmtree(run_artifact_dir)
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(values_format="d")
    plt.title(f"Confusion matrix — {run_name}")
    plt.tight_layout()
    plt.savefig(run_artifact_dir / "confusion_matrix.png", dpi=150)
    plt.close()

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    save_json(run_artifact_dir / "classification_report.json", report)

    save_json(run_artifact_dir / "feature_columns.json", list(feature_cols))
    save_json(run_artifact_dir / "dataset_metadata_snapshot.json", metadata)

    save_json(
        run_artifact_dir / "confusion_matrix_values.json",
        {
            "labels": [0, 1],
            "matrix": cm.tolist(),
        },
    )

    return run_artifact_dir


## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

### MLflow model logging compatibility note

The course tracking server may not support the newer MLflow logged-model API used by some recent MLflow clients. The helper below first tries normal `mlflow.sklearn.log_model`, using `name="model"` when the installed MLflow version supports it to avoid the `artifact_path` deprecation warning. If the server returns a compatibility error, it logs the fitted sklearn pipeline as a `model.joblib` artifact instead, so every run still contains a reproducible model artifact.


In [10]:
def log_sklearn_model_compatible(pipeline, X_example, local_model_dir):
    """Log a fitted sklearn pipeline.

    Recent MLflow clients prefer `name="model"` instead of
    `artifact_path="model"` for model logging. Older clients may not have the
    `name` argument yet, so this helper chooses the supported argument at
    runtime. If the tracking server does not support the newer logged-model
    endpoint, the fitted pipeline is still saved as a joblib artifact.
    Because I don't like seeing warnings😁
    """
    try:
        log_model_kwargs = {
            "sk_model": pipeline,
            "input_example": X_example,
        }

        log_model_parameters = inspect.signature(mlflow.sklearn.log_model).parameters
        if "name" in log_model_parameters:
            log_model_kwargs["name"] = "model"
        else:
            log_model_kwargs["artifact_path"] = "model"

        mlflow.sklearn.log_model(**log_model_kwargs)
        mlflow.set_tag("model_logging_mode", "mlflow_sklearn_model")
        return "mlflow_sklearn_model"

    except Exception as exc:
        message = str(exc)
        fallback_dir = Path(local_model_dir) / "model_fallback"
        fallback_dir.mkdir(parents=True, exist_ok=True)

        model_path = fallback_dir / "model.joblib"
        info_path = fallback_dir / "model_logging_fallback_reason.txt"

        joblib.dump(pipeline, model_path)
        info_path.write_text(
            "mlflow.sklearn.log_model failed, so the fitted sklearn pipeline "
            "was saved as a joblib artifact instead.\n\n"
            f"Error:\n{message}\n",
            encoding="utf-8",
        )

        mlflow.log_artifacts(str(fallback_dir), artifact_path="model_fallback")
        mlflow.set_tag("model_logging_mode", "joblib_artifact_fallback")
        mlflow.set_tag("model_logging_error_type", type(exc).__name__)

        print("MLflow sklearn model logging failed; saved model.joblib as fallback artifact.")
        return "joblib_artifact_fallback"


def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    """Fit, evaluate, and log one complete experiment run to MLflow."""
    pipeline.fit(X_train, y_train)

    metrics, y_pred, y_score = evaluate_binary_classifier(
        pipeline,
        X_test,
        y_test,
        threshold=threshold,
    )

    run_artifact_dir = save_run_artifacts(
        run_name=run_name,
        y_true=y_test,
        y_pred=y_pred,
        feature_cols=feature_cols,
        metadata=metadata,
    )

    params_to_log = {
        **model_params,
        "threshold": threshold,
        "feature_count": len(feature_cols),
        "dataset_version": metadata.get("dataset_version", DATASET_VERSION),
        "dataset_sha256": metadata.get("dataset_sha256", "unknown"),
        "train_rows": int(X_train.shape[0]),
        "test_rows": int(X_test.shape[0]),
    }

    tags_to_log = {
        **tags,
        "dataset_path": metadata.get("dataset_path", ""),
        "notebook": "02_mlflow_experiments_student.ipynb",
    }

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params_to_log)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(tags_to_log)
        mlflow.log_artifacts(str(run_artifact_dir), artifact_path="artifacts")

        model_logging_mode = log_sklearn_model_compatible(
            pipeline=pipeline,
            X_example=X_test.head(5),
            local_model_dir=run_artifact_dir,
        )

        result = {
            "run_id": run.info.run_id,
            "run_name": run_name,
            **metrics,
            "threshold": threshold,
            "model_logging_mode": model_logging_mode,
        }

    print(run_name, result)
    return result

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [11]:
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_preprocessor, leaky_numeric_cols, leaky_categorical_cols = make_preprocessor(X_leaky)

leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

run_results = []

run_results.append(
    run_mlflow_experiment(
        run_name="v0_leaky_logistic_regression",
        pipeline=leaky_pipeline,
        X_train=X_leaky_train,
        X_test=X_leaky_test,
        y_train=y_leaky_train,
        y_test=y_leaky_test,
        feature_cols=leaky_feature_cols,
        model_params={
            "model_family": "logistic_regression",
            "max_iter": 1000,
            "class_weight": "none",
            "random_state": RANDOM_STATE,
            "uses_feature": LEAKAGE_COLUMN,
        },
        tags={
            "leakage_status": "leaky",
            "known_defect": f"uses {LEAKAGE_COLUMN}",
            "model_family": "logistic_regression",
            "experiment_type": "intentional_leakage_check",
        },
    )
)


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/10/runs/b614452153ba4db1a498fdc36da2e192
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v0_leaky_logistic_regression {'run_id': 'b614452153ba4db1a498fdc36da2e192', 'run_name': 'v0_leaky_logistic_regression', 'accuracy': 0.9990458015267175, 'precision': 0.9993746091307066, 'recall': 0.9993746091307066, 'f1': 0.9993746091307066, 'roc_auc': 0.999997483336542, 'threshold': 0.5, 'model_logging_mode': 'joblib_artifact_fallback'}


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [12]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]
)

run_results.append(
    run_mlflow_experiment(
        run_name="v1_dummy_baseline",
        pipeline=dummy_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model_family": "dummy_classifier",
            "strategy": "most_frequent",
        },
        tags={
            "leakage_status": "clean",
            "model_family": "dummy_classifier",
            "experiment_type": "baseline",
        },
    )
)


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/10/runs/aded9221b6b84563bb8c36f24bc23fa3
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v1_dummy_baseline {'run_id': 'aded9221b6b84563bb8c36f24bc23fa3', 'run_name': 'v1_dummy_baseline', 'accuracy': 0.762881679389313, 'precision': 0.762881679389313, 'recall': 1.0, 'f1': 0.8654939106901218, 'roc_auc': 0.5, 'threshold': 0.5, 'model_logging_mode': 'joblib_artifact_fallback'}


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [13]:
clean_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

run_results.append(
    run_mlflow_experiment(
        run_name="v2_clean_logistic_regression",
        pipeline=clean_logistic_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model_family": "logistic_regression",
            "max_iter": 1000,
            "class_weight": "none",
            "random_state": RANDOM_STATE,
        },
        tags={
            "leakage_status": "clean",
            "model_family": "logistic_regression",
            "experiment_type": "clean_real_model",
        },
    )
)


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/10/runs/7128c625edf74004a6f9b2c521346d6e
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v2_clean_logistic_regression {'run_id': '7128c625edf74004a6f9b2c521346d6e', 'run_name': 'v2_clean_logistic_regression', 'accuracy': 0.9742366412213741, 'precision': 0.9777365491651205, 'recall': 0.9887429643527205, 'f1': 0.9832089552238806, 'roc_auc': 0.989391005193135, 'threshold': 0.5, 'model_logging_mode': 'joblib_artifact_fallback'}


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [14]:
balanced_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

run_results.append(
    run_mlflow_experiment(
        run_name="v3_balanced_logistic_regression",
        pipeline=balanced_logistic_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model_family": "logistic_regression",
            "max_iter": 1000,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        },
        tags={
            "leakage_status": "clean",
            "model_family": "logistic_regression",
            "experiment_type": "class_weight_comparison",
        },
    )
)


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/10/runs/f2808725e259472b9158b9dc63e0f278
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v3_balanced_logistic_regression {'run_id': 'f2808725e259472b9158b9dc63e0f278', 'run_name': 'v3_balanced_logistic_regression', 'accuracy': 0.9780534351145038, 'precision': 0.9831985065339142, 'recall': 0.9881175734834271, 'f1': 0.9856519026824704, 'roc_auc': 0.988215723358286, 'threshold': 0.5, 'model_logging_mode': 'joblib_artifact_fallback'}


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [15]:
thresholds = [0.30, 0.40, 0.50, 0.60]

for threshold in thresholds:
    threshold_pipeline = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(X_clean)[0]),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    run_results.append(
        run_mlflow_experiment(
            run_name=f"v4_threshold_{threshold:.2f}_balanced_logistic",
            pipeline=threshold_pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            feature_cols=clean_feature_cols,
            model_params={
                "model_family": "logistic_regression",
                "max_iter": 1000,
                "class_weight": "balanced",
                "random_state": RANDOM_STATE,
                "threshold_strategy": "manual_grid",
            },
            tags={
                "leakage_status": "clean",
                "model_family": "logistic_regression",
                "experiment_type": "threshold_tuning",
            },
            threshold=threshold,
        )
    )


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v4_threshold_0.30_balanced_logistic at: http://185.50.38.163:33014/#/experiments/10/runs/d78e73d5020d45b199794312e4461eef
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v4_threshold_0.30_balanced_logistic {'run_id': 'd78e73d5020d45b199794312e4461eef', 'run_name': 'v4_threshold_0.30_balanced_logistic', 'accuracy': 0.9756679389312977, 'precision': 0.9795539033457249, 'recall': 0.9887429643527205, 'f1': 0.9841269841269841, 'roc_auc': 0.988215723358286, 'threshold': 0.3, 'model_logging_mode': 'joblib_artifact_fallback'}
MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v4_threshold_0.40_balanced_logistic at: http://185.50.38.163:33014/#/experiments/10/runs/f3b1b569a7c04c5b93579fafc1d2cb10
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v4_threshold_0.40_balanced_logistic {'run_id': 'f3b1b569a7c04c5b93579fafc1d2cb10', 'run_name'

## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [16]:
rf_params = {
    "n_estimators": 250,
    "max_depth": 10,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
}

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(X_clean)[0]),
        ("model", RandomForestClassifier(**rf_params, n_jobs=-1)),
    ]
)

run_results.append(
    run_mlflow_experiment(
        run_name="v5_random_forest",
        pipeline=rf_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model_family": "random_forest",
            **rf_params,
            "n_jobs": -1,
        },
        tags={
            "leakage_status": "clean",
            "model_family": "random_forest",
            "experiment_type": "nonlinear_model",
        },
    )
)


MLflow sklearn model logging failed; saved model.joblib as fallback artifact.
🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/10/runs/2f4885cc6b2b42baa62c279937ea4401
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/10
v5_random_forest {'run_id': '2f4885cc6b2b42baa62c279937ea4401', 'run_name': 'v5_random_forest', 'accuracy': 0.9823473282442748, 'precision': 0.9905778894472361, 'recall': 0.9862414008755472, 'f1': 0.9884048887496083, 'roc_auc': 0.992473917929088, 'threshold': 0.5, 'model_logging_mode': 'joblib_artifact_fallback'}


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve the experiment runs.

Important: MLflow may contain failed runs from earlier interrupted executions. For example, a run can fail if the local MLflow client tries to use an endpoint that the course server does not support. Those failed runs are useful for debugging, but they should not be included in model comparison or final model selection.

The comparison table below therefore keeps only runs with:

```text
status = FINISHED
non-null f1
non-null roc_auc
```

Compare at least:

```text
run name
status
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as the final candidate.


In [17]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(f"Experiment not found: {EXPERIMENT_NAME}")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

if runs_df.empty:
    raise ValueError("No MLflow runs were found for this experiment. Run the training cells first.")

# Keep a small audit table of failed/interrupted runs, but exclude them from model comparison.
if "status" in runs_df.columns:
    failed_runs_df = runs_df[~runs_df["status"].eq("FINISHED")].copy()
    finished_runs_df = runs_df[runs_df["status"].eq("FINISHED")].copy()
else:
    failed_runs_df = pd.DataFrame()
    finished_runs_df = runs_df.copy()

wanted_cols = [
    "run_id",
    "status",
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "tags.experiment_type",
    "tags.model_logging_mode",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc",
    "params.threshold",
    "params.dataset_version",
    "params.dataset_sha256",
]

available_cols = [col for col in wanted_cols if col in finished_runs_df.columns]
comparison_df = finished_runs_df[available_cols].copy()

comparison_df = comparison_df.rename(
    columns={
        "tags.mlflow.runName": "run_name",
        "tags.leakage_status": "leakage_status",
        "tags.model_family": "model_family",
        "tags.experiment_type": "experiment_type",
        "tags.model_logging_mode": "model_logging_mode",
        "metrics.accuracy": "accuracy",
        "metrics.precision": "precision",
        "metrics.recall": "recall",
        "metrics.f1": "f1",
        "metrics.roc_auc": "roc_auc",
        "params.threshold": "threshold",
        "params.dataset_version": "dataset_version",
        "params.dataset_sha256": "dataset_sha256",
    }
)

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
for col in metric_cols:
    if col in comparison_df.columns:
        comparison_df[col] = pd.to_numeric(comparison_df[col], errors="coerce")

# Failed or partially logged runs usually have missing metrics. They are excluded here.
required_metric_cols = [col for col in ["f1", "roc_auc"] if col in comparison_df.columns]
if required_metric_cols:
    comparison_df = comparison_df.dropna(subset=required_metric_cols)

if comparison_df.empty:
    raise ValueError(
        "No finished runs with complete metrics were found. "
        "Rerun the experiment cells using the compatibility-fixed logging helper."
    )

comparison_df = comparison_df.sort_values(
    by=["f1", "roc_auc"],
    ascending=False,
    na_position="last",
)

print(f"Total runs found: {len(runs_df)}")
print(f"Finished runs used for comparison: {len(comparison_df)}")
print(f"Failed/interrupted runs ignored: {len(failed_runs_df)}")

comparison_df


Total runs found: 9
Finished runs used for comparison: 9
Failed/interrupted runs ignored: 0


,run_id,status,run_name,leakage_status,model_family,experiment_type,model_logging_mode,accuracy,precision,recall,f1,roc_auc,threshold,dataset_version,dataset_sha256
0,b614452153ba4db1a498fdc36da2e192,FINISHED,v0_leaky_logistic_regression,leaky,logistic_regression,intentional_leakage_check,joblib_artifact_fallback,0.999046,0.999375,0.999375,0.999375,0.999997,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
1,2f4885cc6b2b42baa62c279937ea4401,FINISHED,v5_random_forest,clean,random_forest,nonlinear_model,joblib_artifact_fallback,0.982347,0.990578,0.986241,0.988405,0.992474,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
2,ab323648ed7544a7b5cb01f6e31954f1,FINISHED,v4_threshold_0.60_balanced_logistic,clean,logistic_regression,threshold_tuning,joblib_artifact_fallback,0.980439,0.986875,0.987492,0.987183,0.988216,0.6,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
3,861bf368c75c4d078df4f7b4f621436f,FINISHED,v4_threshold_0.50_balanced_logistic,clean,logistic_regression,threshold_tuning,joblib_artifact_fallback,0.978053,0.983199,0.988118,0.985652,0.988216,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
4,f2808725e259472b9158b9dc63e0f278,FINISHED,v3_balanced_logistic_regression,clean,logistic_regression,class_weight_comparison,joblib_artifact_fallback,0.978053,0.983199,0.988118,0.985652,0.988216,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
5,f3b1b569a7c04c5b93579fafc1d2cb10,FINISHED,v4_threshold_0.40_balanced_logistic,clean,logistic_regression,threshold_tuning,joblib_artifact_fallback,0.977576,0.981988,0.988743,0.985354,0.988216,0.4,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
6,d78e73d5020d45b199794312e4461eef,FINISHED,v4_threshold_0.30_balanced_logistic,clean,logistic_regression,threshold_tuning,joblib_artifact_fallback,0.975668,0.979554,0.988743,0.984127,0.988216,0.3,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
7,7128c625edf74004a6f9b2c521346d6e,FINISHED,v2_clean_logistic_regression,clean,logistic_regression,clean_real_model,joblib_artifact_fallback,0.974237,0.977737,0.988743,0.983209,0.989391,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...
8,aded9221b6b84563bb8c36f24bc23fa3,FINISHED,v1_dummy_baseline,clean,dummy_classifier,baseline,joblib_artifact_fallback,0.762882,0.762882,1.000000,0.865494,0.500000,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...


In [18]:
# Optional audit of failed or interrupted runs.
# These are not used for comparison or final model selection.
failed_wanted_cols = [
    "run_id",
    "status",
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "metrics.f1",
    "metrics.roc_auc",
]

if not failed_runs_df.empty:
    failed_available_cols = [col for col in failed_wanted_cols if col in failed_runs_df.columns]
    failed_audit_df = failed_runs_df[failed_available_cols].copy().rename(
        columns={
            "tags.mlflow.runName": "run_name",
            "tags.leakage_status": "leakage_status",
            "tags.model_family": "model_family",
            "metrics.f1": "f1",
            "metrics.roc_auc": "roc_auc",
        }
    )
    display(failed_audit_df)
else:
    print("No failed/interrupted runs found.")


No failed/interrupted runs found.


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [19]:
clean_candidates = comparison_df[
    comparison_df["leakage_status"].eq("clean")
    & comparison_df["f1"].notna()
    & comparison_df["roc_auc"].notna()
].copy()

if clean_candidates.empty:
    raise ValueError("No finished clean candidate runs with complete metrics were found.")

clean_candidates = clean_candidates.sort_values(
    by=["f1", "roc_auc"],
    ascending=False,
    na_position="last",
)

BEST_RUN_ID = clean_candidates.iloc[0]["run_id"]
best_run_summary = clean_candidates.iloc[0].to_dict()

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")
client.set_tag(BEST_RUN_ID, "selection_rule", "highest_f1_then_roc_auc_among_finished_clean_runs")

print("Selected best finished clean run:", BEST_RUN_ID)
pd.DataFrame([best_run_summary])


Selected best finished clean run: 2f4885cc6b2b42baa62c279937ea4401


,run_id,status,run_name,leakage_status,model_family,experiment_type,model_logging_mode,accuracy,precision,recall,f1,roc_auc,threshold,dataset_version,dataset_sha256
0,2f4885cc6b2b42baa62c279937ea4401,FINISHED,v5_random_forest,clean,random_forest,nonlinear_model,joblib_artifact_fallback,0.982347,0.990578,0.986241,0.988405,0.992474,0.5,v1_student,169297cef9bdfe9ba22b1153ab4514c22cb261d575aef9...


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [20]:
best_run_name = best_run_summary.get("run_name", BEST_RUN_ID)
best_f1 = best_run_summary.get("f1", np.nan)
best_auc = best_run_summary.get("roc_auc", np.nan)
best_precision = best_run_summary.get("precision", np.nan)
best_recall = best_run_summary.get("recall", np.nan)

final_explanation = f"""
I selected {best_run_name} as the final candidate because it was the best clean run by the main comparison criteria, especially F1 and ROC-AUC.
Its test metrics were F1={best_f1:.4f}, ROC-AUC={best_auc:.4f}, precision={best_precision:.4f}, and recall={best_recall:.4f}.
I rejected the leaky run even if its score was high, because it used future_available_rate_30d, which belongs to the future label window and would not be available at prediction time.
The selected model uses a full sklearn Pipeline with preprocessing and the estimator logged together, so it is reproducible for the next serving stage.
As a next step, I would try more threshold tuning, model calibration, and a stronger tree/boosting model while keeping the same clean feature restrictions.
""".strip()

print(final_explanation)


I selected v5_random_forest as the final candidate because it was the best clean run by the main comparison criteria, especially F1 and ROC-AUC.
Its test metrics were F1=0.9884, ROC-AUC=0.9925, precision=0.9906, and recall=0.9862.
I rejected the leaky run even if its score was high, because it used future_available_rate_30d, which belongs to the future label window and would not be available at prediction time.
The selected model uses a full sklearn Pipeline with preprocessing and the estimator logged together, so it is reproducible for the next serving stage.
As a next step, I would try more threshold tuning, model calibration, and a stronger tree/boosting model while keeping the same clean feature restrictions.
